# Ducklake experiment
Ducklake is a new data lake system built on top of DuckDB, designed to provide a simple and efficient way to manage and query large datasets. It leverages the power of DuckDB's SQL engine while adding features for data lake management.

My goal is to explore Ducklake's capabilities, 

## Ducklake documentation
For more information on Ducklake, including installation and usage instructions, please refer to the official documentation

<https://ducklake.select/docs/stable/duckdb/introduction>

In [1]:
import duckdb

In [2]:
duckdb.install_extension("ducklake")
con = duckdb.connect()

In [3]:
con.execute("ATTACH 'ducklake:my_ducklake.ducklake' (DATA_PATH 'ducklake_data/');")

In [6]:
con.execute("CREATE TABLE IF NOT EXISTS my_ducklake.nl_train_stations AS FROM 'https://blobs.duckdb.org/nl_stations.csv';")

In [30]:
con.execute("SELECT * FROM my_ducklake.nl_train_stations").df().sample(10)

,id,code,uic,name_short,name_medium,name_long,slug,country,type,geo_lat,geo_lng
345,723,MTN,8400449,Maastr. N,Maastricht N.,Maastricht Noord,maastricht-noord,NL,stoptreinstation,50.870870,5.717740
419,399,RHN,8400517,Rhenen,Rhenen,Rhenen,rhenen,NL,knooppuntStoptreinstation,51.958611,5.578333
76,810,GSB,8007799,Berlin Gsb,Berlin Gesundbr.,Berlin Gesundbrunnen,berlin-gesundbrunnen,D,intercitystation,52.548633,13.390427
439,436,SPTZ,8400544,Santprt Z,Santpoort Z.,Santpoort Zuid,santpoort-zuid,NL,stoptreinstation,52.419724,4.631389
567,539,ZVB,8400737,Zevenbergn,Zevenbergen,Zevenbergen,zevenbergen,NL,stoptreinstation,51.640420,4.609040
371,713,LNI,8824240,Niel,Niel,Niel,niel,B,stoptreinstation,51.111540,4.338450
480,459,TPSW,8400600,Passewaaij,Passewaaij,Tiel Passewaaij,tiel-passewaaij,NL,stoptreinstation,51.873890,5.392222
534,842,WELS,8101081,Wels Hbf,Wels Hbf,Wels Hbf,wels-hbf,A,intercitystation,48.166111,14.026667
84,77,BHV,8400114,Bilthoven,Bilthoven,Bilthoven,bilthoven,NL,stoptreinstation,52.130001,5.203889
155,145,DTC,8400177,Doetinchem,Doetinchem,Doetinchem,doetinchem,NL,knooppuntStoptreinstation,51.958596,6.296215


In [33]:
con.execute("FROM my_ducklake.snapshots();").df()

,snapshot_id,snapshot_time,schema_version,changes
0,0,2025-06-21 22:45:24.223000+02:00,0,{'schemas_created': ['main']}
1,3,2025-06-24 07:19:25.356000+02:00,1,{}


In [20]:
con.execute("""FROM ducklake_list_files('my_ducklake', 'nl_train_stations');""").df()

,data_file,data_file_size_bytes,data_file_footer_size,data_file_encryption_key,delete_file,delete_file_size_bytes,delete_file_footer_size,delete_file_encryption_key
0,ducklake_data/main/nl_train_stations//ducklake...,60165,1456,<NA>,None,<NA>,<NA>,<NA>


In [15]:
con.execute("INSERT INTO my_ducklake.nl_train_stations VALUES (0, 'KHP', 0, 'khp', 'knut', 'Knut H Pedersen', 'knut_h_pedersen', 'no', 'abc', 59.0, 11.0);")

In [19]:
con.execute("CALL my_ducklake.merge_adjacent_files();")

In [28]:
con.execute("CALL ducklake_expire_snapshots('my_ducklake', versions => [1, 2]);").fetchall()

[(1,
  datetime.datetime(2025, 6, 24, 6, 58, 37, 41000, tzinfo=<DstTzInfo 'Europe/Oslo' CEST+2:00:00 DST>),
  1,
  {'tables_created': ['main.nl_train_stations'],
   'tables_inserted_into': ['1']}),
 (2,
  datetime.datetime(2025, 6, 24, 7, 17, 21, 65000, tzinfo=<DstTzInfo 'Europe/Oslo' CEST+2:00:00 DST>),
  1,
  {'tables_inserted_into': ['1']})]

In [32]:
con.execute("CALL ducklake_cleanup_old_files('my_ducklake', cleanup_all => true);").fetchall()

[('ducklake_data/main/nl_train_stations//ducklake-0197a04d-6912-7c86-a996-6f3842224be1.parquet',),
 ('ducklake_data/main/nl_train_stations//ducklake-0197a05e-8f2c-7420-a485-3902662897e1.parquet',)]